Load packages

In [1]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import Polygon
import math

import maup

import networkx as nx

from gerrychain import (GeographicPartition, Partition, Graph, MarkovChain,
                        proposals, updaters, constraints, accept, Election)
from gerrychain.proposals import recom, propose_random_flip
from functools import partial
import pandas
from gerrychain.tree import recursive_tree_part

from gerrychain.metrics import efficiency_gap, mean_median, polsby_popper, partisan_bias

import os

from gerrychain.constraints.contiguity import contiguous_components, contiguous

from gerrychain.updaters import cut_edges

from gerrychain.tree import bipartition_tree, find_balanced_edge_cuts_memoization

from gerrychain.updaters import num_spanning_trees

import numpy as np

import pandas as pd

import random

Load files

In [ ]:
CON = gpd.read_file("./FL_Processed/fl_cong_adopted_2022/P000C0109.shp")
Precincts = gpd.read_file("./FL_Processed/output/FL_Processed_Precincts_eveomett.shp")

graph = Graph.from_json("./FL_Processed/output/FL_Processed_Precincts_eveomett.json")

C:\Users\angel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\gerrychain\graph\graph.py:406: UserWarning: Found islands (degree-0 nodes). Indices of islands: {934, 1927, 298, 436, 4697}
  warnings.warn(


# Compactness
Setup

In [3]:
Precincts.keys()

Index(['COUNTY20', 'PRECINCT20', 'PCT_STD20', 'SEND', 'HDIST', 'TOTPOP',
       'NH_BLACK', 'NH_OTHER', 'NH_WHITE', 'HISP', 'VAP', 'HVAP', 'WVAP',
       'BVAP', 'OTHERVAP', 'AGR18D', 'AGR18R', 'ATG18D', 'ATG18O', 'ATG18R',
       'CFO18D', 'CFO18O', 'CFO18R', 'GOV18D', 'GOV18O', 'GOV18R', 'PRE16D',
       'PRE16O', 'PRE16R', 'PRE20D', 'PRE20O', 'PRE20R', 'USS16D', 'USS16O',
       'USS16R', 'USS18D', 'USS18O', 'USS18R', 'CD', 'geometry'],
      dtype='object')

In [3]:
# Data conventions: A point is a pair of floats (x, y). A circle is a triple of floats (center x, center y, radius).

# Returns the smallest circle that encloses all the given points. Runs in expected O(n) time, randomized.
# Input: A sequence of pairs of floats or ints, e.g. [(0,5), (3.1,-2.7)].
# Output: A triple of floats representing a circle.
# Note: If 0 points are given, None is returned. If 1 point is given, a circle of radius 0 is returned.
# 
# Initially: No boundary points known
def make_circle(points):
	# Convert to float and randomize order
	shuffled = [(float(x), float(y)) for (x, y) in points]
	random.shuffle(shuffled)
	
	# Progressively add points to circle or recompute circle
	c = None
	for (i, p) in enumerate(shuffled):
		if c is None or not is_in_circle(c, p):
			c = _make_circle_one_point(shuffled[ : i + 1], p)
	return c


# One boundary point known
def _make_circle_one_point(points, p):
	c = (p[0], p[1], 0.0)
	for (i, q) in enumerate(points):
		if not is_in_circle(c, q):
			if c[2] == 0.0:
				c = make_diameter(p, q)
			else:
				c = _make_circle_two_points(points[ : i + 1], p, q)
	return c


# Two boundary points known
def _make_circle_two_points(points, p, q):
	circ = make_diameter(p, q)
	left = None
	right = None
	px, py = p
	qx, qy = q
	
	# For each point not in the two-point circle
	for r in points:
		if is_in_circle(circ, r):
			continue
		
		# Form a circumcircle and classify it on left or right side
		cross = _cross_product(px, py, qx, qy, r[0], r[1])
		c = make_circumcircle(p, q, r)
		if c is None:
			continue
		elif cross > 0.0 and (left is None or _cross_product(px, py, qx, qy, c[0], c[1]) > _cross_product(px, py, qx, qy, left[0], left[1])):
			left = c
		elif cross < 0.0 and (right is None or _cross_product(px, py, qx, qy, c[0], c[1]) < _cross_product(px, py, qx, qy, right[0], right[1])):
			right = c
	
	# Select which circle to return
	if left is None and right is None:
		return circ
	elif left is None:
		return right
	elif right is None:
		return left
	else:
		return left if (left[2] <= right[2]) else right


def make_circumcircle(p0, p1, p2):
	# Mathematical algorithm from Wikipedia: Circumscribed circle
	ax, ay = p0
	bx, by = p1
	cx, cy = p2
	ox = (min(ax, bx, cx) + max(ax, bx, cx)) / 2.0
	oy = (min(ay, by, cy) + max(ay, by, cy)) / 2.0
	ax -= ox; ay -= oy
	bx -= ox; by -= oy
	cx -= ox; cy -= oy
	d = (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by)) * 2.0
	if d == 0.0:
		return None
	x = ox + ((ax * ax + ay * ay) * (by - cy) + (bx * bx + by * by) * (cy - ay) + (cx * cx + cy * cy) * (ay - by)) / d
	y = oy + ((ax * ax + ay * ay) * (cx - bx) + (bx * bx + by * by) * (ax - cx) + (cx * cx + cy * cy) * (bx - ax)) / d
	ra = math.hypot(x - p0[0], y - p0[1])
	rb = math.hypot(x - p1[0], y - p1[1])
	rc = math.hypot(x - p2[0], y - p2[1])
	return (x, y, max(ra, rb, rc))


def make_diameter(p0, p1):
	cx = (p0[0] + p1[0]) / 2.0
	cy = (p0[1] + p1[1]) / 2.0
	r0 = math.hypot(cx - p0[0], cy - p0[1])
	r1 = math.hypot(cx - p1[0], cy - p1[1])
	return (cx, cy, max(r0, r1))


_MULTIPLICATIVE_EPSILON = 1 + 1e-14

def is_in_circle(c, p):
	return c is not None and math.hypot(p[0] - c[0], p[1] - c[1]) <= c[2] * _MULTIPLICATIVE_EPSILON


# Returns twice the signed area of the triangle defined by (x0, y0), (x1, y1), (x2, y2).
def _cross_product(x0, y0, x1, y1, x2, y2):
	return (x1 - x0) * (y2 - y0) - (y1 - y0) * (x2 - x0)


def _continuous_area(geo):
    """returns geo.area"""
    
    return geo.area

def _continuous_perimeter(geo):
    """returns geo.length"""
    
    return geo.length
from math import pi, sqrt
from shapely.geometry.polygon import orient

def perimeter(geo, geo_cell = None, gross = False):
    """
    Return perimeters of geometries in GeoSeries as Series of floats.
    
    Keyword arguments:
        geo -- GeoSeries or GeoDataFrame
        geo_cell -- GeoSeries or GeoDataFrame representing units used to build
            geo (the "container"); does not have to nest cleanly
        gross -- Calculate perimeter of geo by measuringstraight line distance 
            between nodes of geo_cell on boundary of geo (see Schwartzberg 1966)
            
    This function calculates continuous or discrete perimeter. 
    
    Continuous (Euclidean) perimeter is calculated if only geo argument is 
    provided. Currently this function just returns GeoSeries.length. 
    Future improvements could include:
        
        * Checking for lat-long coordinate system and performing geodetic
        measurement
        * Determining appropriate local CRS (most likely a State Plane or UTM
        zone) and performing calculation in that CRS.
        
    NOT YET OPERATIONALIZED: Discrete perimeter is calculated if a second
    geographic argument is provided that represents the "cells" or "building 
    blocks" of the first, larger geography.
    """

    if geo_cell is None:
        # Continuous perimeter
        return _continuous_perimeter(geo)
    elif gross:
        return _gross_perimeter(geo, geo_cell)
    else:
        return _discrete_perimeter(geo, geo_cell)

def area(geo, geo_cell = None, convex_hull = False):
    """
    Return areas of geometries in GeoSeries as Series of floats.
    
    Keyword arguments:
        geo -- GeoSeries or GeoDataFrame
        geo_cell -- GeoSeries or GeoDataFrame representing units used to build
            geo (the "container"); does not have to nest cleanly
        convex_hull -- Calculate area of convex hull of geo
        
    This function calculates continuous or area. 
    
    Continuous (Euclidean) area is calculated if only geo argument is 
    provided. Currently this function just returns GeoSeries.area. 
    Future improvements could include:
        
        * Checking for lat-long coordinate system and performing geodetic
        measurement
        * Determining appropriate local CRS (most likely a State Plane or UTM
        zone) and performing calculation in that CRS.
        
    NOT YET OPERATIONALIZED: Discrete area is calculated if a second
    geographic argument is provided that represents the "cells" or "building 
    blocks" of the first, larger geography.
    """

    if geo_cell is None:
        # Continuous area
        if convex_hull:
            return _continuous_area(geo.convex_hull)
        else:        
            return _continuous_area(geo)
    else:
        return _discrete_area(geo)
    
def polsby_popper_shape(geo, geo_cell = None):
    """
    Returns Polsby-Popper (1991) compactness of geo as float
    
    Keyword arguments:
        geo -- GeoSeries or GeoDataFrame
        geo_cell -- GeoSeries or GeoDataFrame representing units used to build
            geo (the "container"); does not have to nest cleanly
    """
    
    return 4 * pi * area(geo, geo_cell) / (perimeter(geo, geo_cell) ** 2)

def schwartzberg(geo, inverse = True, geo_cell = None):
    """
    Returns Schwartzberg (1966) compactness of geo as float
    
    Keyword arguments:
        geo -- GeoSeries or GeoDataFrame
        inverse -- Boolean, return the original (1 to infinity) Schwartzberg,
            or the inverse (0 to 1); most analysts use the inverse, so this
            function defaults to inverse = True
        geo_cell -- GeoSeries or GeoDataFrame representing units used to build
            geo (the "container"); does not have to nest cleanly
    """
    
    schw = polsby_popper(geo, geo_cell) ** -0.5
    
    if inverse:
        return schw ** -1
    else:
        return schw

def c_hull_ratio(geo):
    
    return area(geo) / area(geo, convex_hull = True)

def reock(geo):
    """
    Returns Reock (1961) compactness of geo as float
    
    Keyword arguments:
        geo -- GeoSeries or GeoDataFrame
    """
    
    # make_circle returns x, y, r of circle. Use r to calculate area.
    mbc_area = geo.convex_hull.apply(lambda x: pi * make_circle(list(x.exterior.coords))[2] ** 2)
    return geo.area / mbc_area

def _polar_moment_of_area(pts, centroid = None):
    """Returns the polar moment of area of a shape
    
    Keyword arguments:
        pts -- Iterable of tuples containing xy coordinate pairs of points
            that define a closed linear ring (i.e. a polygon boundary)
    
    The polar moment of area is the sum of the second moment of area with 
    respect to the x-axis (I_x) and the y-axis (I_y). The moments are 
    calculated with respect to the centroid of the shape.
    
    Adapted from https://leancrew.com/all-this/2018/01/python-module-for-section-properties/
    """
    
    x = [ c[0] for c in pts ]
    y = [ c[1] for c in pts ]

    s = 0
    sx = sy = 0
    sxx = syy = 0 #sxy = 0

    for i in range(len(pts) - 1):
        s += x[i]*y[i+1] - x[i+1]*y[i]
        sx += (x[i] + x[i+1])*(x[i]*y[i+1] - x[i+1]*y[i])
        sy += (y[i] + y[i+1])*(x[i]*y[i+1] - x[i+1]*y[i])
        sxx += (y[i]**2 + y[i]*y[i+1] + y[i+1]**2)*(x[i]*y[i+1] - x[i+1]*y[i])
        syy += (x[i]**2 + x[i]*x[i+1] + x[i+1]**2)*(x[i]*y[i+1] - x[i+1]*y[i])
        # sxy += (x[i]*y[i+1] + 2*x[i]*y[i] + 2*x[i+1]*y[i+1] + x[i+1]*y[i])*(x[i]*y[i+1] - x[i+1]*y[i])

    a = s/2
    if centroid:
        cx, cy = centroid[0], centroid[1]
    else:
        cx, cy = sx/(6*a), sy/(6*a)

    mi = (sxx/12 - a*cy**2) + (syy/12 - a*cx**2)  

    return mi

def _mass_moment_of_inertia(geo, geo_cell, wt):
    """
    Returns the mass moment of inertia of a shape
    
    Keyword arguments:
        geo -- GeoSeries or GeoDataFrame
        geo_cell -- GeoSeries or GeoDataFrame representing units used to build
            geo (the "container"); does not have to nest cleanly
        wt -- Str: Name of column in geo_cell to use as the weight of the unit 
            (e.g. population count) in the moment of inertia calculation
    
    Each geo_cell is assigned to a container geo based on a representative point,
    a point guaranteed to be in geo_cell. The mass of each geo_cell is assumed
    to fall at its centroid. If geo_cell does not nest cleanly (that is, it 
    overlaps neighboring geos), the mass is is nonetheless assigned entirely to
    one container geo.
    """

    # Copy geo_cell so that it is not modified during function    
    tmp = geo_cell[[wt, geo_cell.geometry.name]].copy()
    
    # Calculate area of each cell, used later
    tmp["dA"] = geo_cell.area
    
    # Assign container ID using representative point, since centroid of small
    # geometry may fall outside the geometry, and therefore outside the container
    # for small geometries near edge of container
    tmp.geometry = geo_cell.representative_point()
    tmp.crs = geo_cell.crs # For some reason representative_point() destroys CRS, reassign it
    tmp = gpd.sjoin(tmp, geo[[geo.geometry.name]]) # After sjoin, order of GeoDataFrame changes. Index is preserved.

    # Replace representative point with centroid, which is needed for MI
    tmp.geometry = geo_cell.centroid

    # Calculate population-weighted centroid of each container geo
    # index_right refers to index of each feature in container
    cx = tmp.groupby("index_right").apply(lambda row: (row.geometry.x * row[wt]).sum() / row[wt].sum()).rename("cx")
    cy = tmp.groupby("index_right").apply(lambda row: (row.geometry.y * row[wt]).sum() / row[wt].sum()).rename("cy")
    tmp = tmp.merge(cx, on = "index_right").merge(cy, on = "index_right")
    
    # Calculate moment of inertia of each container geo
    tmp["centroid_distance"] = ((tmp.geometry.x - tmp["cx"])**2 + (tmp.geometry.y - tmp["cy"])**2).apply(sqrt)
    mi = tmp.groupby("index_right").apply(lambda row: (row.centroid_distance**2 * row[wt]).sum()).rename("mi")
    
    # Calculate radius of circle with same area as geo
    circle_mi = tmp.groupby("index_right").apply(lambda row: row[wt].sum() * row["dA"].sum() / (2 * pi))
    
    # Calculate compactness as ratio of MI of circle with same area and equal mass distribution to MI of container
    c_mi = circle_mi / mi
    c_mi.rename("c_mi", inplace = True)
      
    return c_mi
    
def moment_of_inertia(geo, geo_cell = None, wt = 1):
    """
    Returns moment of inertia shape index (MacEachren 1985; Li, et al. 2013) of geo as float
    
    Keyword arguments:
        geo -- GeoSeries or GeoDataFrame
        geo_cell -- GeoSeries or GeoDataFrame representing units used to build
            geo (the "container"); does not have to nest cleanly
        wt -- Int, float, or str: The weight of the unit (e.g. population
            count) in the moment of inertia calculation; this is either a
            number applied as the weight of every unit, or the name of a 
            column in geo_cell with weights; if you want the units weighted
            by area, do not supply values for geo_cell or wt
            
    The moment of inertia of a shape changes as the shape is translated in
    coordinate space. The moment of inertia about the centroid is the 
    minimum moment of inertia for a given shape. The MI of a circle of the
    same area as the shape is divided by the MI of the shape to convert to a 
    "shape index".
    
    When geo_cell is missing, the value calculated is knowsn as the "area 
    moment of inertia" or "second moment of area". It is calculated from the
    polygon coordinates. As with other compactness measures, the final value 
    varies from 0 (least compact) to 1 (most compact, a circle).
    
    When geo_cell is supplied, the value calculated is the "mass moment of
    inertia" or, simply, the "moment of inertia". In this case each cell
    is weighted by some value, specified by the parameter wt. For demographic
    data, the "mass" will usually be the cell population. if the mass is the 
    cell area, this is equivalent to the second moment of inertia, and geo_cell
    should just be omitted. Importanly, in calculating the shape index, the 
    reference circle has the same area of the shape *and uniform density*.
    Therefore, unlike most other compactness measures, this shape index is *not* 
    constrained to be between 0 and 1. If the shape has mass (population)
    concentrated near the center of the shape, the shape can be more compact
    than the reference circle, and the return value will be greater than 1.
    """
    
    if geo_cell is None:

        # Initialize list to hold moments of inertia
        moments = []
        
        # Iterate geometry, which may be a multipolygon
        for geom in geo.geometry:
            
            # Initialize moment of inertia
            mi = 0
            
            # Find centroid of geom, possibly multipolygon
            centroid = (geom.centroid.x, geom.centroid.y)
            
            # geom will contain one or more polygons
            for poly in geom:

                # Give polygon positive orientation (clockwise(?)) so moment
                # of inertia is positive and holes are negative
                poly = orient(poly, sign = 1)
                                
                # Add moment of inertia for exterior points of polygon
                mi += _polar_moment_of_area(poly.exterior.coords, centroid)           
                for interior in poly.interiors:
                    
                    # Subtract moment of inertia for each hole in polygon
                    mi += _polar_moment_of_area(interior.coords, centroid)
                        
            # Convert to shape index by comparing to circle of equal area
            compactness_mi = geom.area**2 / (2 * pi * mi)
            
            moments.append(compactness_mi)
            
        return pd.Series(moments, index = geo.index)
    
    else:
        
        # wt is int or float, repeat value in "wt" column
        if type(wt) in [int, float, tuple, list, pd.Series]:
            geo_cell["wt"] = wt
            wt = "wt"

        return _mass_moment_of_inertia(geo, geo_cell, wt)
    
#Functions from David Rice WSU STATS/CISER

from matplotlib.collections import PatchCollection
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import ConvexHull
import matplotlib.pyplot as plt
def district_reock_vis(district_num, district_color, district_edge_color, district_col_name,circle_color, pie_color,pie_edge_color, df):
    #################################
    #district_num: Integer representing district number
    #district_color: String representing the fill color of the district in the circle
    #district_edge_color: String representing color of district edge
    #district_col_name: The name of the column of which the district is containted
    #circle_color: Color of the bounding circle drawn
    #pie_color: Color of pie chart fill
    #pie_edge_color: Color of pie chart border
    #df: the dataframe
    
    #will print images of bounding circle of the district corresponding to district_num and "liquified" district within circle
    #returns numeric reock score corresonding to district_num
    ##################################
    
    #1 isolating district and plotting:
    
    
    
    #Obtaining dataframe of interest
    df_indexed = df[df[district_col_name] == district_num]
    
    
    #Obtaining boundry points
    edges = list(df_indexed.dissolve(by = district_col_name).exterior)
    coord_list= []
    for i in range(len(edges[0].coords)):
        x = edges[0].coords[i][0]
        y = edges[0].coords[i][1]
        coord_list = coord_list + [(x,y)]

    #Using circle function given before 
    center_radius = make_circle(coord_list)
    district_center = Point((center_radius[0],center_radius[1]))
    
    
    #Obtaining hollow circle 
    circle = district_center.buffer(center_radius[2])
    gdf_circle = gpd.GeoDataFrame(geometry=[circle])
    hole_radius_degrees = center_radius[2] - center_radius[2]*(1/50)
    hole = district_center.buffer(hole_radius_degrees)
    result = gdf_circle.difference(hole)


    
    #Plotting the circle
    fig, ax = plt.subplots()
    # Plot the district boundary
    df_indexed.plot(ax=ax, color=district_color, edgecolor=district_edge_color)
    # Plot the circle
    #gpd.GeoSeries(circle).plot(ax=ax, color='white', alpha=0.5,edgecolor = "red")
    #gpd.GeoSeries(inner_circle).plot(ax = ax, color = "white", alpha = 1, edgecolor = "blue")
    result.plot(ax=ax, color=circle_color, alpha=1)
    plt.show()
    
    
    #Getting Reock and plotting
    #Getting reock score
    circle_area_df = gpd.GeoDataFrame(geometry=[circle], crs='EPSG:4326')
    circle_area = (circle_area_df['geometry'].area.iloc[0])
    district_area = (df_indexed.dissolve(by = district_col_name).area.iloc[0])
    reock_score = (district_area/circle_area)

    #Plotting circle
    #######################################################
    

    miny = float(gdf_circle.bounds["miny"][0])
    maxy = float(gdf_circle.bounds["maxy"][0])
    my_nums = np.arange(miny+0.01, maxy-0.01, 0.01)
    area_lists = []

    for i in my_nums:
        pointlist = []
        for j in gdf_circle["geometry"].boundary[0].coords:
            if(j[1]< i):
                pointlist.append(j)
        points = ([Point(coords) for coords in pointlist])
        geo_series = gpd.GeoSeries(points)
        convex_hull_polygon = gpd.GeoDataFrame(geometry=[geo_series.unary_union.convex_hull])
        area_lists.append(float(convex_hull_polygon.area.iloc[0]))

    area_lists = np.array(area_lists)
    #Solving for best height
    height = my_nums[abs(area_lists - float(district_area)) == min(abs(area_lists-float(district_area)))].min()
    pointlist = []
    for i in gdf_circle["geometry"].boundary[0].coords:
        if(i[1]< height):
                pointlist.append(i)
    points = ([Point(coords) for coords in pointlist])
    geo_series = gpd.GeoSeries(points)
    convex_hull_polygon = gpd.GeoDataFrame(geometry=[geo_series.unary_union.convex_hull])
    
    
    
    ######################################################
    
    
    
    #Plotting the circle
    fig, ax = plt.subplots()
    # Plot the district boundary
    convex_hull_polygon.plot(ax=ax, color=pie_color, linewidth=2, label='Convex Hull')

    result.plot(ax=ax, color=circle_color, alpha=1)

    plt.show()
    
    
    

    return(reock_score)
    
    
#Functions from David Rice WSU STATS/CISER




def district_hull_vis(district_num, district_color, district_col_name,  df):
    #################################
    #district_num: Integer representing district number
    #district_color: String representing the fill color of the district in the circle
    #district_edge_color: String representing color of district edge
    #district_col_name: The name of the column of which the district is containted
    #df: the dataframe
    
    #will print images of bounding convex hull corresponding to district_num and "liquified" district within hull
    #returns numeric ratio of district area over convex hull area
    df_indexed = df[df[district_col_name]== district_num]
       
    #Plotting the convex hull 
    my_district = df_indexed.dissolve(by = district_col_name)
    #Obtaining boundry points
    fig, ax = plt.subplots()
    my_district["geometry"].convex_hull.boundary.plot(ax = ax, color = "black")
    df_indexed.plot(ax = ax, color = district_color, edgecolor = "black")
    plt.show()

    ##############################################
    #Filling the convex hull
    ###########################################

    my_boundary = my_district["geometry"].convex_hull.boundary
    my_boundary = my_boundary.to_crs(crs = 4326)

    district_area = (my_district["geometry"].area)
    print(my_boundary.bounds)
    miny = float(my_boundary.bounds["miny"].iloc[0])
    maxy = float(my_boundary.bounds["maxy"].iloc[0])
    my_nums = np.arange(miny+0.01, maxy-0.01, 0.01)
    area_lists = []

    
    


    #Iteratively getting areas of convex hulls for different height cutoffs
    for i in my_nums:
        pointlist = []
        height = i
        for i in my_boundary[district_num].coords:
            if(i[1]< height):
                pointlist.append(i)
        points = ([Point(coords) for coords in pointlist])
        geo_series = gpd.GeoSeries(points)
        convex_hull_polygon = gpd.GeoDataFrame(geometry=[geo_series.unary_union.convex_hull])
        area_lists.append(float(convex_hull_polygon.area.iloc[0]))

    area_lists = np.array(area_lists)
    #Solving for best height
    height = my_nums[abs(area_lists - float(district_area.iloc[0])) == min(abs(area_lists-float(district_area.iloc[0])))].min()
    pointlist = []
    for i in my_boundary[district_num].coords:
        if(i[1]< height):
            pointlist.append(i)
    points = ([Point(coords) for coords in pointlist])
    geo_series = gpd.GeoSeries(points)
    convex_hull_polygon = gpd.GeoDataFrame(geometry=[geo_series.unary_union.convex_hull])
    
        
    
    fig, ax = plt.subplots()


    my_district["geometry"].convex_hull.boundary.plot(ax = ax, color = "black")
    convex_hull_polygon.plot(ax=ax, color=district_color, linewidth=2, label='Convex Hull')
    plt.show()



    hull_area = my_district["geometry"].convex_hull.area.iloc[0]
    return(district_area/hull_area)



In [4]:
def polsby_popper_gdf(gdf):
    return 4*math.pi * gdf.area/(gdf.length**2)

def count_spanning(graph):
    laplacian = nx.laplacian_matrix(graph)
    L = np.delete(np.delete(laplacian.todense(), 0, 0), 1, 1)
    return np.linalg.slogdet(L)[1]

def county_splits(partition, df=Precincts):
    df["current"] = df.index.map(partition.assignment)

    counties = sum(df.groupby("COUNTY20")['current'].nunique()>1)
    return counties

election_names = [
    "PRE",
    "USS",
    "GOV"
]

num_elections = len(election_names)

election_columns = [
  ['PRE20R','PRE20D'],
  ['USS18R','USS18D'],
  ['GOV18R','GOV18D']
]

my_updaters = {
    "population": updaters.Tally("TOTPOP", alias="population"),
    "cut_edges": cut_edges,
    "PP":polsby_popper,
    "county_splits": county_splits
}

elections = [
    Election(
        election_names[i],
        {"Democratic": election_columns[i][1], "Republican": election_columns[i][0]},
    )
    for i in range(num_elections)
]

election_updaters = {election.name: election for election in elections}
for node in graph.nodes():
    graph.nodes()[node]["NON_NH_BLACK"] = graph.nodes()[node]["TOTPOP"] - graph.nodes()[node]["NH_BLACK"]

my_updaters.update({"NH_BLACK": "NH_BLACK", "NON_NH_BLACK": "NON_NH_BLACK"})

# save percentages

my_updaters.update(election_updaters)


## District-level compactness scores

In [6]:
CON_from_Precincts = Precincts.dissolve(by='CD')
SLDU_from_Precincts = Precincts.dissolve(by='SEND')
SLDL_from_Precincts = Precincts.dissolve(by='HDIST')

In [7]:
plans = [CON_from_Precincts,SLDU_from_Precincts,SLDL_from_Precincts]

for plan in plans: 
    plan['% Black'] = plan['NH_BLACK']/plan['TOTPOP']
    plan['PP'] = polsby_popper_gdf(plan)
    plan['CH'] = c_hull_ratio(plan)
    plan['R'] = 0
    plan['LW'] = 0
    plan['P'] = plan.length
    for ind, row in plan.iterrows():
        plan.loc[ind,'R']=(row['geometry'].area/(math.pi * make_circle(list(row['geometry'].convex_hull.exterior.coords))[2]**2))
        
        
        outside = list(row['geometry'].convex_hull.envelope.exterior.coords)

        o_len = max([x[0] for x in outside]) - min([x[0] for x in outside])

        o_wid = max([x[1] for x in outside]) - min([x[1] for x in outside])

        lw = min(o_len/o_wid,o_wid/o_len)
        
        plan.loc[ind,'LW'] = lw

C:\Users\angel\AppData\Local\Temp\ipykernel_25064\2964938161.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.43499162779272055' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  plan.loc[ind,'R']=(row['geometry'].area/(math.pi * make_circle(list(row['geometry'].convex_hull.exterior.coords))[2]**2))
C:\Users\angel\AppData\Local\Temp\ipykernel_25064\2964938161.py:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5830606206204838' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  plan.loc[ind,'LW'] = lw
C:\Users\angel\AppData\Local\Temp\ipykernel_25064\2964938161.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.37066601691546963' has dtype incompatible 

In [8]:
CON_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./FL_Stats/FL_Compactness_CON_eveomett.csv")
SLDU_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./FL_Stats/FL_Compactness_SLDU_eveomett.csv")
SLDL_from_Precincts[['% Black','PP','CH','R','LW','P']].to_csv("./FL_Stats/FL_Compactness_SLDL_eveomett.csv")

# Cut Edges

In [5]:
CONPart = GeographicPartition(graph,"CD",my_updaters)
SLDUPart = GeographicPartition(graph,"SEND",my_updaters)
SLDLPart = GeographicPartition(graph,"HDIST",my_updaters)

In [9]:
for i in range(1,120):
    if not nx.is_connected(graph.subgraph(CONPart.parts[i])):
        print(i)

1
2
14
19
20


KeyError: 29

In [10]:
print(contiguous(CONPart))
print(contiguous(SLDUPart))
print(contiguous(SLDLPart))

False
False
False


In [11]:
ideal_con_population = sum(CONPart["population"].values()) / len(CONPart)
ideal_sldu_population = sum(SLDUPart["population"].values()) / len(SLDUPart)
ideal_sldl_population = sum(SLDLPart["population"].values()) / len(SLDLPart)

proposed_plans = [CONPart, SLDUPart, SLDLPart]
plan_names = ["CON","SLDU","SLDL"]

clist = ['green','hotpink','orange']

In [12]:
summary = [[] for plan in proposed_plans]


for i in range(len(proposed_plans)):
    #print("Dem Seats:", plan_names[i],proposed_plans[i]['PRS'].wins("Democratic"))
    print(plan_names[i])
    
    print("Cut Edges:", plan_names[i],len(proposed_plans[i]['cut_edges']))

    summary[i].append(len(proposed_plans[i]['cut_edges']))
    
    temp = 0
    for dist in proposed_plans[i].parts:
        tgraph = proposed_plans[i].subgraphs[dist]
        temp += count_spanning(tgraph)


    print("Spanning trees:",plan_names[i],temp)
    summary[i].append(temp)
    print("Mean Polsby_Popper:", plan_names[i],np.mean(list(polsby_popper(proposed_plans[i]).values())))
    summary[i].append(np.mean(list(polsby_popper(proposed_plans[i]).values())))
    print("County Splits:", plan_names[i],county_splits(proposed_plans[i]))
    summary[i].append(county_splits(proposed_plans[i]))

    
    print("Mean Population Deviation",plan_names[i],np.mean([abs((x-ideal_con_population))/ideal_con_population for x in list(proposed_plans[i]['population'].values())]))
    summary[i].append(np.mean([abs((x-ideal_con_population))/ideal_con_population for x in list(proposed_plans[i]['population'].values())]))

CON
Cut Edges: CON 1209
Spanning trees: CON -inf
Mean Polsby_Popper: CON 0.29631156741305487
County Splits: CON 17
Mean Population Deviation CON 0.008050402358776593
SLDU
Cut Edges: SLDU 1453
Spanning trees: SLDU -inf
Mean Polsby_Popper: SLDU 0.2973947157046604
County Splits: SLDU 16
Mean Population Deviation SLDU 0.30000000000000004
SLDL
Cut Edges: SLDL 3025
Spanning trees: SLDL -inf
Mean Polsby_Popper: SLDL 0.3220419321063781
County Splits: SLDL 31
Mean Population Deviation SLDL 0.7666666666666667


In [13]:
summary_df = pd.DataFrame(summary, columns=['Cut Edges', 'Spanning Trees', 'Mean PP', 'County Splits', 'Mean Pop Deviation'], index=plan_names)

In [14]:
summary_df

,Cut Edges,Spanning Trees,Mean PP,County Splits,Mean Pop Deviation
CON,1209,-inf,0.296312,17,0.008050
SLDU,1453,-inf,0.297395,16,0.300000
SLDL,3025,-inf,0.322042,31,0.766667


In [15]:
summary_df.to_csv("./FL_Stats/FL_Compactness_Summary_eveomett.csv")

# Partisan Symmetry
Utilizes 2020 Data

In [18]:
#Helper functions
def plot_symmetry_with_mean_overhaul_rb(pvec,mean,xl=0,xu=1,yl=0,yu=1):
    
    n=5000
    l=len(pvec)
    pvec = np.array(sorted(pvec))
    seats = []
    votes = []
    small = pvec[0]
    large = pvec[-1]
    
    gap = large - small
    
    
    
    #mean = np.mean(pvec)
    
    lvec = pvec - mean*np.ones([1,l])
    
    
    
    for t in range(n):
        tvec = lvec + (t/n)*np.ones([1,l])
        votes.append(t/n)#votes.append(np.mean(tvec))
        seats.append(sum(sum(tvec>=.5))/l)
        
    
 
    #print(lvec,tvec,seats[400])
    
    plt.figure(figsize=(8,8) )   
    plt.plot(votes,seats,linewidth = 3,color='r')
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')

    #plt.plot([.5],[.5],'ro', markersize=10)
    plt.plot([mean],[sum(pvec>=.5)/l],'r*', markersize=20)

    plt.xlabel("Dem Vote %")
    plt.ylabel("Dem Seat %")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)
    
    plt.xlim([xl,xu])
    plt.ylim([yl,yu])

    plt.title("Seats -- Votes")
    
    

    plt.show()
    
    fvotes = [1-x for x in votes]
    fseats = [1-x for x in seats]
    
    plt.figure(figsize=(8,8) )   
    plt.plot(votes,seats,linewidth = 1,color='blue')
    plt.plot(fvotes,fseats,linewidth = 1,color='red')
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')
    plt.fill_between(votes,seats,list(reversed(fseats)),color='gray')

    #plt.plot([.5],[.5],'ro', markersize=10)
    #plt.plot([mean],[sum(pvec>=.5)/l],'g*', markersize=10)

    plt.xlabel("Dem Vote %")
    plt.ylabel("Dem Seat %")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)
    
    plt.xlim([xl,xu])
    plt.ylim([yl,yu])

    plt.title("Seats -- Votes: Symmetry Gaps")    
  

def plot_lots_symmetry_notmean(pvecs,means,xl=0,xu=1,yl=0,yu=1):
    
    n=5000
    
    ind = 0 
    
    clist = ['tab:blue','tab:orange','tab:green','tab:red','tab:purple','tab:brown',
            'tab:pink','tab:gray','tab:olive','tab:cyan','black','lime','navy','burlywood',
            'salmon','blueviolet','chocolate','yellow','fuchsia','silver']
    plt.figure(figsize=(8,8) )  
    for pvec in pvecs: 
        l=len(pvec)
        pvec = np.array(sorted(pvec))
        seats = []
        votes = []
        small = pvec[0]
        large = pvec[-1]

        gap = large - small



        mean = means[ind] #np.mean(pvec)

        lvec = pvec - mean*np.ones([1,l])



        for t in range(n):
            tvec = lvec + (t/n)*np.ones([1,l])
            votes.append(np.mean(tvec))
            seats.append(sum(sum(tvec>=.5))/l)


        bn = np.array([mean+ (.5-x) for x in pvec])


        cn=[str(round(float(bn), 3)) for bn in bn]
        bvotes=[]
        bseats=[]

        for t in range(n):
            bvotes.append(t/n)
            bseats.append(sum(bn<(t/n))/l)


        bn = np.array([mean+ (.5-x) for x in pvec])

        bvotes=[]
        bseats=[]

        for t in range(n):
            bvotes.append(t/n)
            bseats.append(sum(bn<(t/n))/l)

        dn=list(bn[:])
        for x in bn:
            dn.append(1-x)

        rseats=list(reversed(seats))


        en=[str(round(float(dn), 3)) for dn in dn]
        area=0
        for t in range(n):
            area += (1/n)*abs(seats[t]-(1-rseats[t])) 



         
        plt.plot(bvotes,bseats,linewidth = 2,color=clist[ind],label=enames[ind],alpha=.5)
        

        #plt.plot([.5],[.5],'ro', markersize=10)
        plt.plot([mean],[sum(pvec>=.5)/l],'o',color=clist[ind], markersize=8,zorder=100)
        
        ind +=1
    
    
    plt.axhline(.5,color='gray')
    plt.axvline(.5,color='gray')
    plt.xlabel("Democratic Vote Share")
    plt.ylabel("Democratic Seat Share")
    #plt.xticks(bn,cn, rotation=45)
    ys=[x/l for x in range(l+1)]
    zs=[str(round(float(ys), 3)) for ys in ys]
    #plt.yticks(ys,zs)

    plt.xlim([xl,xu])
    plt.ylim([yl,yu])
    plt.legend()

    #plt.title("Seats -- Votes")
    
    

    plt.show()

 
def declination(vals):
    """ Compute the declination of an election.
    """
    Rwin = sorted(filter(lambda x: x <= 0.5, vals))
    Dwin = sorted(filter(lambda x: x > 0.5, vals))
    # Undefined if each party does not win at least one seat
    if len(Rwin) < 1 or len(Dwin) < 1:
        return False
    theta = np.arctan((1-2*np.mean(Rwin))*len(vals)/len(Rwin))
    gamma = np.arctan((2*np.mean(Dwin)-1)*len(vals)/len(Dwin))
    # Convert to range [-1,1]
    # A little extra precision just in case.
    return 2.0*(gamma-theta)/3.1415926535 


def lopsided_wins(vals):
    Rwin = sorted(filter(lambda x: x <= 0.5, vals))
    Dwin = sorted(filter(lambda x: x > 0.5, vals))
    
    Rmargin = abs(np.mean(Rwin)-.5)*2
    Dmargin = abs(np.mean(Dwin)-.5)*2
    
    return Rmargin - Dmargin
    

In [16]:
plan_partitions = proposed_plans
plan_part_labels = plan_names
enames = election_names

In [20]:
n_base_plans = 3

wins = [[] for i in range(n_base_plans)]
votes = [[] for i in range(n_base_plans)]
majs = [[] for i in range(n_base_plans)]
mms = [[] for i in range(n_base_plans)]
egs = [[] for i in range(n_base_plans)]
pbs =[[] for i in range(n_base_plans)]
decs = [[] for i in range(n_base_plans)]
lws = [[] for i in range(n_base_plans)]
comps = [[] for i in range(n_base_plans)]


for election in range(num_elections):
    summary = [[] for election in range(n_base_plans)]

    print(enames[election])
    #print('Votes: ', plan_partitions[0][enames[election]].percent("Democratic"))
    
    for i in range(n_base_plans):
        votes[i].append(plan_partitions[i][enames[election]].percent("Democratic"))
        summary[i].append(plan_partitions[i][enames[election]].percent("Democratic"))
        print('Votes: ', plan_partitions[i][enames[election]].percent("Democratic"))
    print('Seats')

    for i in range(n_base_plans):
        wins[i].append(plan_partitions[i][enames[election]].wins("Democratic"))
        print(plan_part_labels[i],wins[i][-1])
        summary[i].append(plan_partitions[i][enames[election]].wins("Democratic"))
    
    print("Majority?")
    for i in range(n_base_plans):
    
        if plan_partitions[i][enames[election]].percent("Democratic") > .5:
            if plan_partitions[i][enames[election]].wins("Democratic")>len(plan_partitions[i])/2:
                majs[i].append(1)
            else:
                majs[i].append(0)
        else:
            if plan_partitions[i][enames[election]].wins("Democratic")>len(plan_partitions[i])/2:
                majs[i].append(0)
            else:
                majs[i].append(1)
        print(plan_part_labels[i],majs[i][-1])
        summary[i].append(majs[i][-1])
    
    print("Mean-Median")
    for i in range(n_base_plans):
        mms[i].append(np.median(plan_partitions[i][enames[election]].percents("Democratic"))-plan_partitions[i][enames[election]].percent("Democratic"))
        print(plan_part_labels[i],mms[i][-1])
        summary[i].append(mms[i][-1])
        
    print("Partisan Bias")
    for i in range(n_base_plans):
        pbs[i].append(partisan_bias(plan_partitions[i][enames[election]]))
        print(plan_part_labels[i],pbs[i][-1])
        summary[i].append(pbs[i][-1])
    
    print("Efficiency Gap")
    for i in range(n_base_plans):
        egs[i].append(efficiency_gap(plan_partitions[i][enames[election]]))
        print(plan_part_labels[i],egs[i][-1])
        summary[i].append(egs[i][-1])
    
    
    print("Declination")
    for i in range(n_base_plans):
        decs[i].append(declination(plan_partitions[i][enames[election]].percents("Democratic")))
        print(plan_part_labels[i],decs[i][-1])
        summary[i].append(decs[i][-1])
        
    print("Lopsided Wins")
    for i in range(n_base_plans):
        lws[i].append(lopsided_wins(plan_partitions[i][enames[election]].percents("Democratic")))
        print(plan_part_labels[i],lws[i][-1])
        summary[i].append(lws[i][-1])
        
        
    print("Competitive 47-53 Districts")
    for i in range(n_base_plans):
        comps[i].append(np.sum([.47 < x <.53 for x in plan_partitions[i][enames[election]].percents("Democratic") ]))
        print(plan_part_labels[i],comps[i][-1])
        summary[i].append(comps[i][-1])
    
    df_summary = pd.DataFrame(summary, columns=['Votes', 'Seats', 'Majority', 'Mean-Median', 'Partisan Bias', 'Efficiency Gap', 'Declination', 'Lopsided Wins', 'Competitive Districts'], index=plan_part_labels)
    display(df_summary)
    df_summary.to_csv("./FL_Stats/FL_Partisan_Summary_"+enames[election]+"_eveomett.csv")
        

PRE
Votes:  0.48305245337858443
Votes:  0.48305245337858443
Votes:  0.48305245337858443
Seats
CON 8
SLDU 16
SLDL 50
Majority?
CON 1
SLDU 1
SLDL 1
Mean-Median
CON -0.02262333832416824
SLDU -0.03582594173072501
SLDL -0.01133421301617138
Partisan Bias
CON -0.17857142857142855
SLDU -0.07500000000000001
SLDL -0.07500000000000001
Efficiency Gap
CON -0.20174149098066566
SLDU -0.09214249862481233
SLDL -0.08277152478766664
Declination
CON 0.3679495510863084
SLDU 0.14080401032289722
SLDL 0.1395332520160032
Lopsided Wins
CON -0.13767512059011477
SLDU -0.04515924627104917
SLDL -0.0553327583907145
Competitive 47-53 Districts
CON 3
SLDU 5
SLDL 22


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.483052,8,1,-0.022623,-0.178571,-0.201741,0.367950,-0.137675,3
SLDU,0.483052,16,1,-0.035826,-0.075000,-0.092142,0.140804,-0.045159,5
SLDL,0.483052,50,1,-0.011334,-0.075000,-0.082772,0.139533,-0.055333,22


USS
Votes:  0.49927395381234424
Votes:  0.49927395381234424
Votes:  0.49927395381234424
Seats
CON 10
SLDU 19
SLDL 55
Majority?
CON 1
SLDU 1
SLDL 1
Mean-Median
CON -0.026938239234098937
SLDU -0.030442304715654733
SLDL -0.011693148474691883
Partisan Bias
CON -0.14285714285714285
SLDU -0.04999999999999999
SLDL -0.08333333333333331
Efficiency Gap
CON -0.1781653180329593
SLDU -0.05615785520885756
SLDL -0.06922409572346244
Declination
CON 0.2974189581974398
SLDU 0.08366742995506962
SLDL 0.1385980258176219
Lopsided Wins
CON -0.15045011232758254
SLDU -0.05637610786333769
SLDL -0.09424136239828729
Competitive 47-53 Districts
CON 4
SLDU 5
SLDL 22


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.499274,10,1,-0.026938,-0.142857,-0.178165,0.297419,-0.150450,4
SLDU,0.499274,19,1,-0.030442,-0.050000,-0.056158,0.083667,-0.056376,5
SLDL,0.499274,55,1,-0.011693,-0.083333,-0.069224,0.138598,-0.094241,22


GOV
Votes:  0.4979556286297231
Votes:  0.49795562862972303
Votes:  0.497955628629723
Seats
CON 10
SLDU 18
SLDL 54
Majority?
CON 1
SLDU 1
SLDL 1
Mean-Median
CON -0.026898670303825978
SLDU -0.029697300072660193
SLDL -0.010353784609139716
Partisan Bias
CON -0.14285714285714285
SLDU -0.07500000000000001
SLDL -0.09999999999999998
Efficiency Gap
CON -0.1727889244439672
SLDU -0.07969254487467216
SLDL -0.07621238616374887
Declination
CON 0.29518280156964355
SLDU 0.12500514909086327
SLDL 0.15156803883536135
Lopsided Wins
CON -0.14747748899723256
SLDU -0.07323328886063352
SLDL -0.09911012090929128
Competitive 47-53 Districts
CON 4
SLDU 5
SLDL 22


,Votes,Seats,Majority,Mean-Median,Partisan Bias,Efficiency Gap,Declination,Lopsided Wins,Competitive Districts
CON,0.497956,10,1,-0.026899,-0.142857,-0.172789,0.295183,-0.147477,4
SLDU,0.497956,18,1,-0.029697,-0.075000,-0.079693,0.125005,-0.073233,5
SLDL,0.497956,54,1,-0.010354,-0.100000,-0.076212,0.151568,-0.099110,22
